In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("Incremental Flag","0")

In [0]:
increm_flag=dbutils.widgets.get("Incremental Flag")
print(increm_flag)

In [0]:
df_src=spark.sql('''
        select * from parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/icd10_diagnosis`''')

In [0]:
df_src.limit(5).display()

In [0]:
if spark.catalog.tableExists("healthcare.gold.dim_diagnosis"):
    df_sink=spark.sql('''
        select * from healthcare.gold.dim_diagnosis''')
else:
    df_sink=spark.sql('''
            select 1 as dim_diagnosis_key,icd10_code,icd10_description,diagnosis_category,chronic_flag,code_set_version from parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/icd10_diagnosis` where 1=0
        ''')

In [0]:
df_sink.limit(5).display()

In [0]:
df_filter=df_src.join(df_sink,df_src["icd10_code"]==df_sink["icd10_code"],"left").select(df_src["*"],df_sink["dim_diagnosis_key"])

In [0]:
df_filter.limit(5).display()

In [0]:
df_new_record=df_filter.filter(df_filter["dim_diagnosis_key"].isNull())
df_existing_record=df_filter.filter(df_filter["dim_diagnosis_key"].isNotNull())

In [0]:
df_new_record.limit(5).display()

In [0]:
df_existing_record.limit(5).display()

In [0]:
if increm_flag == '0':
    max_value = 1
else:
    max_value_df = spark.sql('''
                          select coalesce(max(dim_diagnosis_key),0 ) from healthcare.gold.dim_diagnosis
                          ''')
    max_value = max_value_df.collect()[0][0]+1

In [0]:
windowSpec=Window.orderBy(col("icd10_code"))
df_new_record=df_new_record.withColumn("dim_diagnosis_key",row_number().over(windowSpec)+lit(max_value-1))


In [0]:
df_new_record.limit(5).display()

In [0]:
df_final=df_new_record.unionByName(df_existing_record)
df_final.limit(5).display()

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("healthcare.gold.dim_diagnosis"):
    deltaTable=DeltaTable.forName(spark,"healthcare.gold.dim_diagnosis")
    deltaTable.alias("target").merge(df_final.alias("source"),"target.dim_diagnosis_key=source.dim_diagnosis_key")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
else:
    df_final.write.format("delta")\
        .mode("overwrite")\
            .saveAsTable("healthcare.gold.dim_diagnosis")

In [0]:
%sql
select * from healthcare.gold.dim_diagnosis;